# Smart-meter Data Exploration

## Imports

In [19]:
import os
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from IPython.display import clear_output

## Load data

### Directories and filepaths

After more processed data has been generated, add filepaths here.

In [2]:
base_dir = os.path.join(os.getcwd(), '..')
processed_data_dir = os.path.join(base_dir, 'processed_data')

ausgrid_filepath = os.path.join(processed_data_dir, 'ausgrid_processed_data.csv')
open_meteo_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')

### Load data

After more processed data has been generated, load it here.

In [3]:
ausgrid_df = pd.read_csv(ausgrid_filepath, parse_dates=['datetime'])
open_meteo_df = pd.read_csv(open_meteo_filepath, parse_dates=['datetime'])

## Prepare Data

In [4]:
df = pd.merge(ausgrid_df, open_meteo_df, on="datetime", how="inner")

# Get customers
customers = sorted(df['Customer'].dropna().unique())

# Get available numberic variables
variables = [
    col for col in df.columns
    if col not in ["datetime", "date", "time", "Customer"]
    and pd.api.types.is_numeric_dtype(df[col])]

#### Construct day-of-year and hour-of-day numeric columns

In [5]:
df['date'] = df['datetime'].dt.date
df['hour'] = df["datetime"].dt.hour
df["day_of_year"] = df["datetime"].dt.dayofyear  # 1–365 (or 366 in leap years)
df["hour_of_day"] = df["datetime"].dt.hour              # 0–23

## Daily Profile Dashboard

### Construct interactive controls

In [ ]:
# Customer dropdown
customer_dropdown = widgets.Dropdown(
    options=customers,
    description='Customer',
    layout=widgets.Layout(width='400px'))

# Date dropdown
date_dropdown = widgets.Dropdown(
    description='Date:',
    layout=widgets.Layout(width='400px'))

# Variable dropdown
variable_dropdown = widgets.Dropdown(
    options=variables,
    description='Variable:',
    layout=widgets.Layout(width='400px'))

# Normalize toggle
normalize_toggle = widgets.ToggleButton(
    value=False,
    description='Normalize: OFF',
    button_style='',
    layout=widgets.Layout(width="150px"))

# Add profile button
add_button = widgets.Button(
    description='Add profile',
    button_style='primary',
    layout=widgets.Layout(width='150px'))

# Clear profile button
clear_button = widgets.Button(
    description='Clear profiles',
    layout=widgets.Layout(width='150px'))

output = widgets.Output()

### Define functions for updating the dashboard

#### Update dates
Update the available dates when the customer changes.

In [7]:
def update_dates(change):
    customer = change['new']
    customer_df = df[df['Customer'] == customer]
    dates = sorted(customer_df['date'].unique())
    
    date_dropdown.options = dates
    if dates: date_dropdown.value = dates[0]

#### Update toggle label

In [8]:
def update_normalize_button(change):
    if change['new']:
        normalize_toggle.description = 'Normalize: ON'
        normalize_toggle.button_style = 'success'
    else:
        normalize_toggle.description = 'Normalize: OFF'
        normalize_toggle.button_style = ''

### Update plot

Draw all currently selected profiles

In [9]:
def update_plot(change=None):

    with output:
        clear_output(wait=True)
        plt.figure(figsize=(10, 5))

        # Plot every profile currently stored
        for customer, date, variable, normalized in profiles:
            day_df = df[(df['Customer'] == customer) & (df['date'] == date)].sort_values('datetime')
            if day_df.empty: continue

            y = day_df[variable].copy()

            if normalized:
                y_min = y.min()
                y_max = y.max()
                if y_max != y_min:
                    y = (y - y_min) / (y_max - y_min)
                else: y = 0

            normalization_label = " (normalized)" if normalized else ""
            plt.plot(day_df['hour'], y, label=f"Customer {customer} - {date} - {variable}{normalization_label}")

        plt.xlabel("Time of day")
        # No Y-axis label because multiple variables can be plotted.

        if profiles:
            plt.title("Selected profiles")
            plt.legend()
        else:
            plt.title("No profiles selected")

        plt.grid(True)
        plt.tight_layout()
        plt.show()

#### Add profile

In [10]:
def add_profile(button):
    button.description = "Adding..."
    button.disabled = True

    try:
        customer = customer_dropdown.value
        date = date_dropdown.value
        variable = variable_dropdown.value
        normalized = normalize_toggle.value

        if customer is None or date is None or variable is None:
            return

        profile = (customer, date, variable, normalized)

        # Don't add the same profile more than once.
        if profile not in profiles:
            profiles.append(profile)
            update_plot()
    finally:
        button.description = "Add profile"
        button.disabled = False

##### Clear profiles

In [11]:
def clear_profiles(change=None):
    profiles.clear()
    update_plot()

### Connect and initialize the dashboard

In [ ]:
profiles = []
add_button = widgets.Button(description="Add profile")

# Connect widgets
customer_dropdown.observe(update_dates, names='value')
date_dropdown.observe(update_plot, names='value')
variable_dropdown.observe(update_plot, names='value')
normalize_toggle.observe(update_normalize_button, names='value')

add_button.on_click(add_profile)
clear_button.on_click(clear_profiles)

# Initialize the date dropdown
update_dates({'new': customer_dropdown.value})

# Display dashboard
display(
    widgets.VBox([
        customer_dropdown,
        date_dropdown,
        widgets.HBox([
            variable_dropdown,
            normalize_toggle
        ]),
        widgets.HBox([
            add_button,
            clear_button
        ])
    ]),
    output)

# Initial empty plot
update_plot()

Output()

## Scatterplot Dashboard
### Prepare Data

In [14]:
variables = [
    col for col in df.columns
        if col not in ["datetime", "Customer"]
        and pd.api.types.is_numeric_dtype(df[col])]

### Construct interactive controls.

In [15]:
# X-axis variable
x_variable_dropdown = widgets.Dropdown(
    options=variables,
    value='Pload' if 'Pload' in variables else variables[0] if variables else None,
    description='X variable:',
    layout=widgets.Layout(width='450px'))

# Y-axis variable
y_variable_dropdown = widgets.Dropdown(
    options=variables,
    value='Pgen' if 'Pgen' in variables else variables[1] if len(variables) > 1 else variables[0] if variables else None,
    description='Y variable:',
    layout=widgets.Layout(width='450px'))

# Customer dropdown
scatter_customer_dropdown = widgets.Dropdown(
    options=customers + ['All customers'],
    value='All customers',
    description='Customer:',
    layout=widgets.Layout(width='450px'))

# Best-fit line dropdown
fit_type_dropdown = widgets.Dropdown(
    options=['Linear'],
    value='Linear',
    description='Fit type:',
    layout=widgets.Layout(width='450px'))

# Best-fit line toggle
fit_button = widgets.Button(
    description="Show best fit",
    button_style="primary",
    layout=widgets.Layout(width='150px'))

### Plotting function

In [16]:
def update_scatterplot(change=None):
    x_variable = x_variable_dropdown.value
    y_variable = y_variable_dropdown.value
    customer = scatter_customer_dropdown.value
    fit_enabled = fit_button.description == 'Hide best fit'

    if x_variable is None or y_variable is None:
        with scatter_output:
            clear_output(wait=True)
        with fit_output:
            clear_output(wait=True)
        return

    plot_df = df if customer == 'All customers' else df[df['Customer'] == customer]
    sample_size = min(len(plot_df), 50000)
    sample_df = plot_df.sample(n=sample_size, random_state=42) if len(plot_df) > sample_size else plot_df
    plot_data = pd.DataFrame({'x': sample_df[x_variable].to_numpy(), 'y': sample_df[y_variable].to_numpy()}).replace([np.inf, -np.inf], np.nan).dropna()
    x = plot_data['x']
    y = plot_data['y']
    fit_error = None
    fit_stats = None

    with scatter_output:
        clear_output(wait=True)
        plt.figure(figsize=(10, 5))
        point_label = f'Data points (sample of {len(plot_df):,})' if len(plot_df) > sample_size else 'Data points'
        plt.scatter(x, y, alpha=0.5, label=point_label)
        if plot_data.empty:
            plt.text(0.5, 0.5, 'No valid numeric data for these variables.', ha='center', va='center', transform=plt.gca().transAxes)

        if fit_enabled:
            if len(plot_data) < 2 or x.nunique() < 2:
                fit_error = 'A linear fit requires at least two valid points with distinct X values.'
            else:
                coefficients = np.polyfit(x, y, 1)
                x_fit = np.linspace(x.min(), x.max(), 200)
                y_fit = np.polyval(coefficients, x_fit)
                plt.plot(x_fit, y_fit, color='red', label='Linear fit')

                y_predicted = np.polyval(coefficients, x)
                ss_res = np.sum((y - y_predicted) ** 2)
                ss_tot = np.sum((y - y.mean()) ** 2)
                r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
                fit_stats = (coefficients[0], coefficients[1], r_squared, len(plot_data), len(plot_df))

        plt.xlabel(x_variable)
        plt.ylabel(y_variable)
        title = f'Scatter plot of {y_variable} vs {x_variable}'
        if customer != 'All customers':
            title += f' - Customer {customer}'
        plt.title(title)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    with fit_output:
        clear_output(wait=True)
        if fit_enabled:
            if fit_error:
                print(fit_error)
            elif fit_stats is not None:
                slope, intercept, r_squared, point_count, total_count = fit_stats
                print('Fit type: Linear')
                print(f'R-squared: {r_squared:.4f}')
                print(f'Equation: y = {slope:.4f}x + {intercept:.4f}')
                print(f'Points used: {point_count:,}' + (f' (sampled from {total_count:,})' if point_count < total_count else ''))

### Best-fit function

In [17]:
def show_best_fit(change=None):
    fit_enabled = fit_button.description != 'Hide best fit'
    fit_button.description = 'Hide best fit' if fit_enabled else 'Show best fit'
    fit_button.button_style = 'warning' if fit_enabled else 'primary'
    update_scatterplot()

### Connect and initialize the dashboard

In [21]:
# Create output areas before connecting the widget callbacks
scatter_output = widgets.Output()
fit_output = widgets.Output()

# Update plot whenever a dropdown changes
x_variable_dropdown.observe(
    update_scatterplot,
    names='value')

y_variable_dropdown.observe(
    update_scatterplot,
    names='value')

scatter_customer_dropdown.observe(
    update_scatterplot,
    names='value')

fit_type_dropdown.observe(
    update_scatterplot,
    names='value')

# Connect best-fit button
fit_button.on_click(show_best_fit)

# Display dashboard
display(
    widgets.VBox([
        x_variable_dropdown,
        y_variable_dropdown,
        scatter_customer_dropdown,
        fit_type_dropdown,
        fit_button,
        scatter_output,
        fit_output
    ]))

# Initial scatterplot
update_scatterplot()